In [ ]:
# ============================================================
# 01_RG3_원본복원.py
#
# 목적
# 1. moldset_labeled_rg3.csv   -> labeled_data.csv 원본값 복원
# 2. moldset_unlabeled_rg3.csv -> unlabeled_data.csv 원본값 복원
# 3. TimeStamp / Side / 생산정보 / 원본 공정값 복원
# 4. 현재 RG3 scaled 데이터와 원본 매칭 검증
# ============================================================

from pathlib import Path

import numpy as np
import pandas as pd


# ============================================================
# 1. 경로
# ============================================================

BASE_DIR = Path(
    r"../../../data/raw"
)

LABELED_RAW_PATH = BASE_DIR / "labeled_data.csv"
UNLABELED_RAW_PATH = BASE_DIR / "unlabeled_data.csv"

RG3_TRAIN_PATH = BASE_DIR / "moldset_labeled_rg3.csv"
RG3_TEST_PATH = BASE_DIR / "moldset_unlabeled_rg3.csv"

TRAIN_OUTPUT_PATH = BASE_DIR / "RG3_train_raw_restored.csv"
TEST_OUTPUT_PATH = BASE_DIR / "RG3_test_raw_restored.csv"


# ============================================================
# 2. 파일 존재 확인
# ============================================================

paths = {
    "labeled_data": LABELED_RAW_PATH,
    "unlabeled_data": UNLABELED_RAW_PATH,
    "RG3 train": RG3_TRAIN_PATH,
    "RG3 test": RG3_TEST_PATH,
}

print("=" * 100)
print("FILE CHECK")
print("=" * 100)

for name, path in paths.items():
    print(f"{name:<20} : {path.exists()} | {path}")

missing = [
    str(path)
    for path in paths.values()
    if not path.exists()
]

if missing:
    raise FileNotFoundError(
        "다음 파일을 찾을 수 없습니다.\n"
        + "\n".join(missing)
    )


# ============================================================
# 3. 데이터 로드
# ============================================================

print("\n" + "=" * 100)
print("LOAD DATA")
print("=" * 100)

labeled_raw = pd.read_csv(
    LABELED_RAW_PATH,
    low_memory=False
)

unlabeled_raw = pd.read_csv(
    UNLABELED_RAW_PATH,
    low_memory=False
)

rg3_train = pd.read_csv(
    RG3_TRAIN_PATH,
    low_memory=False
)

rg3_test = pd.read_csv(
    RG3_TEST_PATH,
    low_memory=False
)

print("labeled_data           :", labeled_raw.shape)
print("unlabeled_data         :", unlabeled_raw.shape)
print("moldset_labeled_rg3    :", rg3_train.shape)
print("moldset_unlabeled_rg3  :", rg3_test.shape)


# ============================================================
# 4. Unnamed: 0 확인
# ============================================================

INDEX_COL = "Unnamed: 0"

if INDEX_COL not in rg3_train.columns:
    raise KeyError(
        f"Train에 '{INDEX_COL}' 컬럼이 없습니다."
    )

if INDEX_COL not in rg3_test.columns:
    raise KeyError(
        f"Test에 '{INDEX_COL}' 컬럼이 없습니다."
    )


# 정수 index로 변환
train_raw_index = (
    pd.to_numeric(
        rg3_train[INDEX_COL],
        errors="raise"
    )
    .astype(int)
)

test_raw_index = (
    pd.to_numeric(
        rg3_test[INDEX_COL],
        errors="raise"
    )
    .astype(int)
)


# ============================================================
# 5. index 범위 검증
# ============================================================

print("\n" + "=" * 100)
print("INDEX CHECK")
print("=" * 100)

print(
    "Train index range:",
    train_raw_index.min(),
    "~",
    train_raw_index.max()
)

print(
    "Labeled raw rows:",
    len(labeled_raw)
)

print(
    "Test index range:",
    test_raw_index.min(),
    "~",
    test_raw_index.max()
)

print(
    "Unlabeled raw rows:",
    len(unlabeled_raw)
)


if train_raw_index.min() < 0:
    raise ValueError("Train에 음수 원본 index가 있습니다.")

if train_raw_index.max() >= len(labeled_raw):
    raise ValueError(
        "Train의 Unnamed: 0 값이 "
        "labeled_data 행 범위를 초과합니다."
    )

if test_raw_index.min() < 0:
    raise ValueError("Test에 음수 원본 index가 있습니다.")

if test_raw_index.max() >= len(unlabeled_raw):
    raise ValueError(
        "Test의 Unnamed: 0 값이 "
        "unlabeled_data 행 범위를 초과합니다."
    )


# ============================================================
# 6. 원본 행 직접 복원
# ============================================================

train_restored = (
    labeled_raw
    .iloc[train_raw_index.to_numpy()]
    .copy()
    .reset_index(drop=False)
    .rename(
        columns={
            "index": "original_index"
        }
    )
)

test_restored = (
    unlabeled_raw
    .iloc[test_raw_index.to_numpy()]
    .copy()
    .reset_index(drop=False)
    .rename(
        columns={
            "index": "original_index"
        }
    )
)


# ============================================================
# 7. 현재 RG3 행 번호도 보존
# ============================================================

train_restored.insert(
    0,
    "rg3_row_index",
    np.arange(len(train_restored))
)

test_restored.insert(
    0,
    "rg3_row_index",
    np.arange(len(test_restored))
)


# ============================================================
# 8. Target 정리
# ============================================================

# 원본 labeled_data의 PassOrFail이
# Y/N 형태라면 1/0으로 변환한 별도 컬럼 생성

if "PassOrFail" in train_restored.columns:

    raw_target = (
        train_restored["PassOrFail"]
        .astype(str)
        .str.strip()
        .str.upper()
    )

    target_map = {
        "Y": 1,
        "N": 0,
        "1": 1,
        "0": 0,
    }

    train_restored[
        "PassOrFail_binary"
    ] = raw_target.map(target_map)


# ============================================================
# 9. Side 생성
# ============================================================

# PART_NAME에서 RH / LH 추출

def extract_side(value):

    if pd.isna(value):
        return pd.NA

    text = str(value).upper()

    if "RH" in text:
        return "RH"

    if "LH" in text:
        return "LH"

    return pd.NA


if "PART_NAME" in train_restored.columns:

    train_restored["Side"] = (
        train_restored["PART_NAME"]
        .apply(extract_side)
    )

if "PART_NAME" in test_restored.columns:

    test_restored["Side"] = (
        test_restored["PART_NAME"]
        .apply(extract_side)
    )


# ============================================================
# 10. TimeStamp 변환
# ============================================================

for df in [
    train_restored,
    test_restored
]:

    if "TimeStamp" in df.columns:

        df["TimeStamp"] = pd.to_datetime(
            df["TimeStamp"],
            errors="coerce"
        )


# ============================================================
# 11. Train Target 매칭 검증
# ============================================================

print("\n" + "=" * 100)
print("TRAIN TARGET CHECK")
print("=" * 100)

if (
    "PassOrFail" in rg3_train.columns
    and
    "PassOrFail_binary"
    in train_restored.columns
):

    scaled_target = (
        pd.to_numeric(
            rg3_train["PassOrFail"],
            errors="coerce"
        )
    )

    raw_target = (
        train_restored[
            "PassOrFail_binary"
        ]
    )

    valid = (
        scaled_target.notna()
        &
        raw_target.notna()
    )

    same = (
        scaled_target[valid]
        .astype(int)
        .to_numpy()
        ==
        raw_target[valid]
        .astype(int)
        .to_numpy()
    )

    print(
        "Target comparison rows:",
        valid.sum()
    )

    print(
        "Target same:",
        same.sum()
    )

    print(
        "Target match rate:",
        same.mean()
    )

    if not same.all():

        print(
            "\n[WARNING]"
            " Target이 일치하지 않는 행이 있습니다."
        )


# ============================================================
# 12. Side / Target 분포 확인
# ============================================================

print("\n" + "=" * 100)
print("TRAIN SIDE / TARGET")
print("=" * 100)

if "Side" in train_restored.columns:

    print("\nSide counts")
    print(
        train_restored[
            "Side"
        ].value_counts(
            dropna=False
        )
    )


if (
    "Side" in train_restored.columns
    and
    "PassOrFail_binary"
    in train_restored.columns
):

    print("\nSide x Target")
    print(
        pd.crosstab(
            train_restored["Side"],
            train_restored[
                "PassOrFail_binary"
            ],
            margins=True
        )
    )


# ============================================================
# 13. TimeStamp 확인
# ============================================================

print("\n" + "=" * 100)
print("TIME RANGE")
print("=" * 100)

for name, df in [
    ("Train", train_restored),
    ("Test", test_restored)
]:

    if "TimeStamp" in df.columns:

        print(
            f"{name}:",
            df["TimeStamp"].min(),
            "~",
            df["TimeStamp"].max()
        )


# ============================================================
# 14. 원본 센서 vs scaled 데이터 구조 확인
# ============================================================

# moldset RG3와 원본에 동시에 있는 수치형 센서
# Target / index 제외

EXCLUDE_COLS = {
    "Unnamed: 0",
    "PassOrFail",
    "original_index",
    "rg3_row_index",
}

common_train_cols = [
    col
    for col in rg3_train.columns
    if (
        col in train_restored.columns
        and
        col not in EXCLUDE_COLS
        and
        pd.api.types.is_numeric_dtype(
            rg3_train[col]
        )
        and
        pd.api.types.is_numeric_dtype(
            train_restored[col]
        )
    )
]


print("\n" + "=" * 100)
print("COMMON PROCESS FEATURES")
print("=" * 100)

print(
    "공통 수치형 공정변수 수:",
    len(common_train_cols)
)

print(common_train_cols)


# ============================================================
# 15. StandardScaler 재현 검증
# ============================================================

# 현재 moldset_labeled_rg3가
# 원본 train 자체 평균/std로 scaling된 것인지 검사
#
# sklearn StandardScaler:
# z = (x - mean) / std(ddof=0)

print("\n" + "=" * 100)
print("TRAIN SCALING RECONSTRUCTION CHECK")
print("=" * 100)

train_scaling_results = []

for col in common_train_cols:

    raw = pd.to_numeric(
        train_restored[col],
        errors="coerce"
    ).astype(float)

    scaled = pd.to_numeric(
        rg3_train[col],
        errors="coerce"
    ).astype(float)

    mean = raw.mean()
    std = raw.std(ddof=0)

    if (
        pd.isna(std)
        or
        np.isclose(std, 0)
    ):

        reconstructed = pd.Series(
            np.zeros(len(raw)),
            index=raw.index
        )

    else:

        reconstructed = (
            raw - mean
        ) / std

    abs_diff = (
        reconstructed.to_numpy()
        -
        scaled.to_numpy()
    )

    abs_diff = np.abs(abs_diff)

    train_scaling_results.append(
        {
            "Feature": col,
            "Raw_Mean": mean,
            "Raw_Std": std,
            "Mean_Abs_Diff": np.nanmean(
                abs_diff
            ),
            "Max_Abs_Diff": np.nanmax(
                abs_diff
            ),
        }
    )


train_scaling_check = pd.DataFrame(
    train_scaling_results
)

print(
    train_scaling_check[
        [
            "Feature",
            "Mean_Abs_Diff",
            "Max_Abs_Diff",
        ]
    ].to_string(
        index=False
    )
)


# ============================================================
# 16. Test scaling 재현 검증
# ============================================================

common_test_cols = [
    col
    for col in rg3_test.columns
    if (
        col in test_restored.columns
        and
        col not in EXCLUDE_COLS
        and
        pd.api.types.is_numeric_dtype(
            rg3_test[col]
        )
        and
        pd.api.types.is_numeric_dtype(
            test_restored[col]
        )
    )
]


print("\n" + "=" * 100)
print("TEST SCALING RECONSTRUCTION CHECK")
print("=" * 100)

test_scaling_results = []

for col in common_test_cols:

    raw = pd.to_numeric(
        test_restored[col],
        errors="coerce"
    ).astype(float)

    scaled = pd.to_numeric(
        rg3_test[col],
        errors="coerce"
    ).astype(float)

    mean = raw.mean()
    std = raw.std(ddof=0)

    if (
        pd.isna(std)
        or
        np.isclose(std, 0)
    ):

        reconstructed = pd.Series(
            np.zeros(len(raw)),
            index=raw.index
        )

    else:

        reconstructed = (
            raw - mean
        ) / std

    abs_diff = np.abs(
        reconstructed.to_numpy()
        -
        scaled.to_numpy()
    )

    test_scaling_results.append(
        {
            "Feature": col,
            "Raw_Mean": mean,
            "Raw_Std": std,
            "Mean_Abs_Diff": np.nanmean(
                abs_diff
            ),
            "Max_Abs_Diff": np.nanmax(
                abs_diff
            ),
        }
    )


test_scaling_check = pd.DataFrame(
    test_scaling_results
)

print(
    test_scaling_check[
        [
            "Feature",
            "Mean_Abs_Diff",
            "Max_Abs_Diff",
        ]
    ].to_string(
        index=False
    )
)


# ============================================================
# 17. 시간순 정렬용 컬럼 생성
# ============================================================

# 원본 행은 그대로 유지하고
# time_order만 추가
# 실제 정렬은 다음 분석 단계에서 수행

if "TimeStamp" in train_restored.columns:

    train_time_order = (
        train_restored["TimeStamp"]
        .sort_values(
            kind="stable"
        )
        .index
    )

    order_map = {
        idx: order
        for order, idx
        in enumerate(train_time_order)
    }

    train_restored[
        "time_order"
    ] = train_restored.index.map(
        order_map
    )


if "TimeStamp" in test_restored.columns:

    test_time_order = (
        test_restored["TimeStamp"]
        .sort_values(
            kind="stable"
        )
        .index
    )

    order_map = {
        idx: order
        for order, idx
        in enumerate(test_time_order)
    }

    test_restored[
        "time_order"
    ] = test_restored.index.map(
        order_map
    )


# ============================================================
# 18. 결과 저장
# ============================================================

train_restored.to_csv(
    TRAIN_OUTPUT_PATH,
    index=False,
    encoding="utf-8-sig"
)

test_restored.to_csv(
    TEST_OUTPUT_PATH,
    index=False,
    encoding="utf-8-sig"
)


# scaling 검증 결과도 저장
train_scaling_check.to_csv(
    BASE_DIR
    / "RG3_train_scaling_check.csv",
    index=False,
    encoding="utf-8-sig"
)

test_scaling_check.to_csv(
    BASE_DIR
    / "RG3_test_scaling_check.csv",
    index=False,
    encoding="utf-8-sig"
)


# ============================================================
# 19. 최종 요약
# ============================================================

print("\n" + "=" * 100)
print("RESTORE COMPLETE")
print("=" * 100)

print(
    "Train restored:",
    train_restored.shape
)

print(
    "Test restored :",
    test_restored.shape
)

print("\n저장 위치")

print(
    "Train:",
    TRAIN_OUTPUT_PATH
)

print(
    "Test :",
    TEST_OUTPUT_PATH
)

print(
    "Train scaling check:",
    BASE_DIR
    / "RG3_train_scaling_check.csv"
)

print(
    "Test scaling check :",
    BASE_DIR
    / "RG3_test_scaling_check.csv"
)


print("\nTrain 주요 컬럼")

check_cols = [
    "original_index",
    "TimeStamp",
    "PART_NAME",
    "Side",
    "PART_FACT_SERIAL",
    "PART_FACT_PLAN_DATE",
    "PassOrFail",
    "PassOrFail_binary",
    "Reason",
]

print(
    [
        col
        for col in check_cols
        if col in train_restored.columns
    ]
)